In [1]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
from optimizer import OrthogonalOptimizer
import gc

In [2]:
wine = fetch_openml(name='wine-quality-white', version=1, as_frame=False)
X, y = wine.data, wine.target.astype(int)
y = y - y.min()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
test_dataset  = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(dataset=train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=30, shuffle=False)

In [3]:
classes, counts = np.unique(y, return_counts=True)
total_samples = len(y)
print("Class distribution")
for cls, count in zip(classes, counts):
    percentage = (count / total_samples) * 100
    print(f"Class {cls}: {percentage:.2f}%")

Class distribution
Class 0: 0.41%
Class 1: 3.33%
Class 2: 29.75%
Class 3: 44.88%
Class 4: 17.97%
Class 5: 3.57%
Class 6: 0.10%


In [4]:
class mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(11, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 7)
        )

    def forward(self, x):
        return self.layers(x)

model = mlp()
criterion = nn.CrossEntropyLoss()
optimizer = OrthogonalOptimizer(model.parameters(), lr=0.01, num_bits=5)
gc.collect()

1084

In [5]:
epochs = 10
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

In [6]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs}]')
    print("Training loss", (running_loss / total))  
    train_losses.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss_eval / total))
    test_losses.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)
    
    gc.collect()

Epoch [1/10]
Training loss 233508773213470.97
Training accuracy 37.110770801429304 %
Testing loss 230346721757142.22
Testing accuracy 36.326530612244895 %
Epoch [2/10]
Training loss 232307898511073.56
Training accuracy 35.83460949464012 %
Testing loss 239093387817043.6
Testing accuracy 33.87755102040816 %
Epoch [3/10]
Training loss 254619502732748.5
Training accuracy 33.052577845839714 %
Testing loss 245366148776939.1
Testing accuracy 33.775510204081634 %
Epoch [4/10]
Training loss 264733001779560.7
Training accuracy 31.5211842776927 %
Testing loss 277906088290429.38
Testing accuracy 32.44897959183673 %
Epoch [5/10]
Training loss 304166222276186.7
Training accuracy 24.706482899438488 %
Testing loss 324520953458437.25
Testing accuracy 22.448979591836736 %
Epoch [6/10]
Training loss 309346176394349.25
Training accuracy 23.838693210821848 %
Testing loss 302652881119691.75
Testing accuracy 18.163265306122447 %
Epoch [7/10]
Training loss 332945189055497.94
Training accuracy 14.8034711587544

In [7]:
class mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(11, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 7)
        )

    def forward(self, x):
        return self.layers(x)

model = mlp()
criterion = nn.CrossEntropyLoss()
optimizer = OrthogonalOptimizer(model.parameters(), lr=0.01, num_bits=5)
gc.collect()

8

In [8]:
epochs = 10
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

In [9]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs}]')
    print("Training loss", (running_loss / total))  
    train_losses.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss_eval / total))
    test_losses.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)
    
    gc.collect()

Epoch [1/10]
Training loss 415972160528062.0
Training accuracy 27.71822358346095 %
Testing loss 486722484209622.2
Testing accuracy 27.755102040816325 %
Epoch [2/10]
Training loss 508980248056596.25
Training accuracy 28.101071975497703 %
Testing loss 493953749175902.06
Testing accuracy 28.163265306122447 %
Epoch [3/10]
Training loss 530341193052375.4
Training accuracy 28.994384890250128 %
Testing loss 585358194764570.1
Testing accuracy 29.183673469387756 %
Epoch [4/10]
Training loss 592822257641627.8
Training accuracy 29.632465543644717 %
Testing loss 606556363653788.8
Testing accuracy 28.877551020408163 %
Epoch [5/10]
Training loss 584946861510323.5
Training accuracy 29.428279734558448 %
Testing loss 569989360650867.0
Testing accuracy 28.877551020408163 %
Epoch [6/10]
Training loss 585848465466954.0
Training accuracy 29.17304747320061 %
Testing loss 627891357587330.6
Testing accuracy 28.979591836734695 %
Epoch [7/10]
Training loss 622314813080530.5
Training accuracy 29.657988769780502 